In [1]:
import sqlite3
import pandas as pd


# ---------------------------
# Database connection helper
# ---------------------------
def connect_db(db_name="lusitania.db"):
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


# ---------------------------
# Class Table
# ---------------------------
class TravelClass:
    @staticmethod
    def create_table(conn):
        conn.execute("""
        CREATE TABLE IF NOT EXISTS class_table (
            class_id INTEGER PRIMARY KEY AUTOINCREMENT,
            class_name TEXT NOT NULL UNIQUE
        );
        """)
        conn.commit()

    @staticmethod
    def add_class(conn, class_name):
        conn.execute("""
        INSERT OR IGNORE INTO class_table (class_name)
        VALUES (?);
        """, (class_name,))
        conn.commit()

    @staticmethod
    def get_all_classes(conn):
        cursor = conn.execute("SELECT * FROM class_table ORDER BY class_id;")
        return cursor.fetchall()

    @staticmethod
    def get_class_id_by_name(conn, class_name):
        cursor = conn.execute("""
        SELECT class_id
        FROM class_table
        WHERE class_name = ?;
        """, (class_name,))
        row = cursor.fetchone()
        return row[0] if row else None


# ---------------------------
# Passenger Table
# ---------------------------
class Passenger:
    @staticmethod
    def create_table(conn):
        conn.execute("""
        CREATE TABLE IF NOT EXISTS passenger (
            passenger_id INTEGER PRIMARY KEY,
            family_name TEXT,
            title TEXT,
            personal_name TEXT,
            age TEXT,
            citizenship TEXT,
            city TEXT,
            adult_minor TEXT,
            sex TEXT,
            status TEXT,
            class_id INTEGER,
            FOREIGN KEY (class_id) REFERENCES class_table(class_id)
                ON DELETE SET NULL
                ON UPDATE CASCADE
        );
        """)
        conn.commit()

    @staticmethod
    def add_passenger(conn, passenger_id, family_name, title, personal_name,
                      age, citizenship, city, adult_minor, sex, status, class_id):
        conn.execute("""
        INSERT OR REPLACE INTO passenger
        (passenger_id, family_name, title, personal_name, age, citizenship,
         city, adult_minor, sex, status, class_id)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """, (
            passenger_id, family_name, title, personal_name, age,
            citizenship, city, adult_minor, sex, status, class_id
        ))
        conn.commit()

    @staticmethod
    def get_all_passengers(conn):
        cursor = conn.execute("""
        SELECT *
        FROM passenger
        ORDER BY passenger_id;
        """)
        return cursor.fetchall()

    @staticmethod
    def list_passengers_with_class(conn):
        cursor = conn.execute("""
        SELECT
            p.passenger_id,
            p.personal_name,
            p.family_name,
            c.class_name
        FROM passenger p
        LEFT JOIN class_table c
            ON p.class_id = c.class_id
        ORDER BY p.passenger_id;
        """)
        return cursor.fetchall()


# ---------------------------
# Department Table
# ---------------------------
class Department:
    @staticmethod
    def create_table(conn):
        conn.execute("""
        CREATE TABLE IF NOT EXISTS department (
            department_id INTEGER PRIMARY KEY AUTOINCREMENT,
            department_name TEXT NOT NULL UNIQUE
        );
        """)
        conn.commit()

    @staticmethod
    def add_department(conn, department_name):
        conn.execute("""
        INSERT OR IGNORE INTO department (department_name)
        VALUES (?);
        """, (department_name,))
        conn.commit()

    @staticmethod
    def get_all_departments(conn):
        cursor = conn.execute("SELECT * FROM department ORDER BY department_id;")
        return cursor.fetchall()

    @staticmethod
    def get_department_id_by_name(conn, department_name):
        cursor = conn.execute("""
        SELECT department_id
        FROM department
        WHERE department_name = ?;
        """, (department_name,))
        row = cursor.fetchone()
        return row[0] if row else None


# ---------------------------
# Crew Table
# ---------------------------
class Crew:
    @staticmethod
    def create_table(conn):
        conn.execute("""
        CREATE TABLE IF NOT EXISTS crew (
            crew_id INTEGER PRIMARY KEY,
            family_name TEXT,
            title TEXT,
            personal_name TEXT,
            age TEXT,
            citizenship TEXT,
            city TEXT,
            adult_minor TEXT,
            sex TEXT,
            position TEXT,
            department_id INTEGER,
            FOREIGN KEY (department_id) REFERENCES department(department_id)
                ON DELETE SET NULL
                ON UPDATE CASCADE
        );
        """)
        conn.commit()

    @staticmethod
    def add_crew(conn, crew_id, family_name, title, personal_name,
                 age, citizenship, city, adult_minor, sex, position, department_id):
        conn.execute("""
        INSERT OR REPLACE INTO crew
        (crew_id, family_name, title, personal_name, age, citizenship,
         city, adult_minor, sex, position, department_id)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """, (
            crew_id, family_name, title, personal_name, age,
            citizenship, city, adult_minor, sex, position, department_id
        ))
        conn.commit()

    @staticmethod
    def get_all_crew(conn):
        cursor = conn.execute("""
        SELECT *
        FROM crew
        ORDER BY crew_id;
        """)
        return cursor.fetchall()

    @staticmethod
    def list_crew_with_department(conn):
        cursor = conn.execute("""
        SELECT
            c.crew_id,
            c.personal_name,
            c.family_name,
            c.position,
            d.department_name
        FROM crew c
        LEFT JOIN department d
            ON c.department_id = d.department_id
        ORDER BY c.crew_id;
        """)
        return cursor.fetchall()


# ---------------------------
# Create all tables
# ---------------------------
def create_all_tables(conn):
    TravelClass.create_table(conn)
    Passenger.create_table(conn)
    Department.create_table(conn)
    Crew.create_table(conn)


# ---------------------------
# Load CSV into database
# ---------------------------
def load_lusitania_data(conn, csv_file):
    df = pd.read_csv(csv_file)

    # Clean column names just in case
    df.columns = [col.strip() for col in df.columns]

    # Separate passengers and crew
    passenger_df = df[df["Passenger/Crew"] == "Passenger"].copy()
    crew_df = df[df["Passenger/Crew"] == "Crew"].copy()

    # ---------------------------
    # Load passenger classes
    # ---------------------------
    passenger_classes = (
        passenger_df["Department/Class"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    for class_name in sorted(passenger_classes):
        TravelClass.add_class(conn, class_name)

    # ---------------------------
    # Load crew departments
    # ---------------------------
    crew_departments = (
        crew_df["Department/Class"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    for dept_name in sorted(crew_departments):
        Department.add_department(conn, dept_name)

    # ---------------------------
    # Load passengers
    # ---------------------------
    for _, row in passenger_df.iterrows():
        class_name = str(row["Department/Class"]).strip() if pd.notna(row["Department/Class"]) else None
        class_id = TravelClass.get_class_id_by_name(conn, class_name) if class_name else None

        Passenger.add_passenger(
            conn=conn,
            passenger_id=int(row["ID"]),
            family_name=row["Family_name"],
            title=row["Title"],
            personal_name=row["Personal_name"],
            age=str(row["Age"]) if pd.notna(row["Age"]) else None,
            citizenship=row["Citizenship"],
            city=row["City"],
            adult_minor=row["Adult/Minor"],
            sex=row["Sex"],
            status=row["Status"],
            class_id=class_id
        )

    # ---------------------------
    # Load crew
    # ---------------------------
    for _, row in crew_df.iterrows():
        dept_name = str(row["Department/Class"]).strip() if pd.notna(row["Department/Class"]) else None
        department_id = Department.get_department_id_by_name(conn, dept_name) if dept_name else None

        Crew.add_crew(
            conn=conn,
            crew_id=int(row["ID"]),
            family_name=row["Family_name"],
            title=row["Title"],
            personal_name=row["Personal_name"],
            age=str(row["Age"]) if pd.notna(row["Age"]) else None,
            citizenship=row["Citizenship"],
            city=row["City"],
            adult_minor=row["Adult/Minor"],
            sex=row["Sex"],
            position=row["Position"],
            department_id=department_id
        )


# ---------------------------
# Example queries
# ---------------------------
def show_counts(conn):
    passenger_count = conn.execute("SELECT COUNT(*) FROM passenger;").fetchone()[0]
    crew_count = conn.execute("SELECT COUNT(*) FROM crew;").fetchone()[0]
    class_count = conn.execute("SELECT COUNT(*) FROM class_table;").fetchone()[0]
    department_count = conn.execute("SELECT COUNT(*) FROM department;").fetchone()[0]

    print("Passenger records:", passenger_count)
    print("Crew records:", crew_count)
    print("Class records:", class_count)
    print("Department records:", department_count)


def show_sample_data(conn):
    print("\nCLASSES")
    print(TravelClass.get_all_classes(conn))

    
    print("\nDEPARTMENTS")
    print(Department.get_all_departments(conn))

    print("\nFIRST 10 PASSENGERS WITH CLASS")
    rows = Passenger.list_passengers_with_class(conn)[:10]
    for row in rows:
        print(row)

    print("\nFIRST 10 CREW WITH DEPARTMENT")
    rows = Crew.list_crew_with_department(conn)[:10]
    for row in rows:
        print(row)

def passenger_older_30(conn):
    cursor = conn.execute("SELECT * FROM passenger WHERE age >= 30;")
    passengers = cursor.fetchall()
    return passengers

    

# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    csv_file = "LusitaniaManifest.csv"   # put the CSV in the same folder as this script
    conn = connect_db("lusitania_2.db")

    create_all_tables(conn)
    load_lusitania_data(conn, csv_file)

    show_counts(conn)
    show_sample_data(conn)

    for row in passenger_older_30(conn):
        print(row)

    conn.close()

Passenger records: 1265
Crew records: 693
Class records: 3
Department records: 4

CLASSES
[(1, 'Saloon'), (2, 'Second'), (3, 'Third')]

DEPARTMENTS
[(1, 'Band'), (2, 'Deck'), (3, 'Engineering'), (4, 'Victualling')]

FIRST 10 PASSENGERS WITH CLASS
(387, 'William McMillan', 'ADAMS', 'Saloon')
(388, 'Arthur Henry', 'ADAMS', 'Saloon')
(389, 'Henry', 'ADAMS', 'Saloon')
(390, 'Henry (Annie Elizabeth Macnutt)', 'ADAMS', 'Saloon')
(391, 'Hugh Montagu (Marguerite Ethel MacKenzie)', 'ALLAN', 'Saloon')
(392, 'Gwendolyn Evelyn', 'ALLAN', 'Saloon')
(393, 'Anna Marjory', 'ALLAN', 'Saloon')
(394, 'Dorothy Ditman (nurse to the Cromptons)', 'ALLEN', 'Saloon')
(395, 'Nicholas Naftel', 'ALLES', 'Saloon')
(396, 'Julian de', 'AYALA', 'Saloon')

FIRST 10 CREW WITH DEPARTMENT
(0, 'Charles W.', 'CAMERON', None, 'Band')
(1, 'E.', 'CARR-JONES', None, 'Band')
(2, 'Edward', 'DRAKEFORD', 'Violin', 'Band')
(3, 'Handel', 'HAWKINS', 'Cello', 'Band')
(4, 'John William', 'HEMINGWAY', 'Double Bass', 'Band')
(5, 'James C